In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
anonymized = os.path.join(path,'Q3_data.csv')
df = pd.read_csv(anonymized)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in numerical_cols:
  df[col] = df[col].fillna(df[col].mean()) # Numbers: use mean
for col in categorical_cols:
  df[col] = df[col].fillna(df[col].mode()[0]) # Categories: use mode


In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=['object']).columns
categorical_cols
# there is not categorical columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
X = df.drop("Target", axis=1)
y = df["Target"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 5: Write your code here:
# Target imbalance means the classes in your target variable are NOT evenly distributed - one class appears WAY more than the other.
# Count each class
print(y.value_counts())

# Show as percentages
print(y.value_counts(normalize=True) * 100)

In [ ]:
# Storage for logistic regression results for each fold
lr_accuracy = []
lr_f1 = []

In [ ]:
# Task 1,2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
# Use previously generated random classification data
model = CatBoostClassifier()
# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_fold_pred = model.predict(X_test)

       # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_fold_pred)
    f1 = f1_score(y_test, y_fold_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
import numpy as np


print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Feature importance
import matplotlib.pyplot as plt

feature_cols = X.select_dtypes(include=['int64', 'float64']).columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(15, 35))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#The golden feature is P_2 and the second one is D_42.

In [ ]:
# Task Bonus: Write your code here: